# TCGA-BRCA Multimodal Preprocessing Pipeline

Downloads RNA-seq, CNV, and DNA Methylation from UCSC Xena, preprocesses each modality, aligns samples, and produces low-rank approximations:

$$X_i = U_i D_i V_i^\top + Z_i, \quad Z_i \sim \mathcal{N}(0, \sigma_i^2)$$

Output matrices $X_1, X_2, X_3$ share the same row index (samples) but have different numbers of columns (features).

In [ ]:
import sys
!{sys.executable} -m pip install requests tqdm scikit-learn pandas numpy scipy matplotlib seaborn --quiet

## 0. Imports

In [ ]:
import os
import requests
import gzip
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## 1. Paths & Configuration

In [ ]:
BASE_DIR = Path(".")  # set this to the local path of this data_preprocessing/ folder (or wherever data/ and matrices/ should live)
DATA_DIR = BASE_DIR / "data"
OUT_DIR  = BASE_DIR / "matrices"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Highly variable feature selection ----
# Number of top features retained per modality (by variance, before z-scoring).
# Reduce these if memory is a constraint.
TOP_FEATURES = {
    "rna":  2000,   # out of ~20 000 genes
    "cnv":  1000,   # out of ~24 000 genes  (discrete ±2 values, fewer needed)
    "meth": 5000,   # out of ~485 000 CpG sites
}

VARIANCE_THRESHOLD = 0.90   # fraction of variance explained for SVD rank selection

print("Data directory :", DATA_DIR)
print("Output directory:", OUT_DIR)
print("Top features    :", TOP_FEATURES)

## 2. Download from UCSC Xena

Files are from the GDC TCGA Breast Cancer hub. Each download is skipped if the file already exists.

In [ ]:
XENA_FILES = {
    "rna": {
        "url": "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FHiSeqV2_PANCAN.gz",
        "filename": "BRCA_RNAseq.tsv.gz",
        "description": "RNA-seq gene expression (RSEM log2 normalized)"
    },
    "cnv": {
        "url": "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FGistic2_CopyNumber_Gistic2_all_data_by_genes.gz",
        "filename": "BRCA_CNV.tsv.gz",
        "description": "CNV GISTIC2 continuous log-ratio (Gaussian-compatible)"
    },
    "meth": {
        "url": "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FHumanMethylation450.gz",
        "filename": "BRCA_Methylation450.tsv.gz",
        "description": "DNA Methylation 450k beta values"
    },
    "survival": {
        "url": "https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/survival%2FBRCA_survival.txt",
        "filename": "BRCA_survival.tsv",
        "description": "Survival / clinical labels"
    }
}


def download_file(url: str, dest: Path, description: str = ""):
    # Skip only if file exists AND is non-empty
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  [SKIP] {dest.name} already exists ({dest.stat().st_size / 1e6:.1f} MB).")
        return
    if dest.exists():
        print(f"  [WARN] {dest.name} exists but is empty — re-downloading.")
        dest.unlink()
    print(f"  Downloading {description} ...")
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    total = int(response.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total or None, unit="B", unit_scale=True, desc=dest.name) as bar:
        for chunk in response.iter_content(chunk_size=65536):
            f.write(chunk)
            bar.update(len(chunk))
    print(f"  Saved: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")


for key, info in XENA_FILES.items():
    download_file(info["url"], DATA_DIR / info["filename"], info["description"])

## 3. Load Raw Data

Xena files are stored as (features × samples) TSVs — we transpose to (samples × features).

In [ ]:
def load_xena_tsv(filepath: Path) -> pd.DataFrame:
    """Load a Xena TSV (features x samples).
    Handles both plain TSV and gzip-compressed files transparently,
    including the case where requests decompressed .gz on the fly.
    """
    import gzip as _gzip
    print(f"  Loading {filepath.name} ...")

    # Detect whether the file is actually gzip-compressed
    with open(filepath, "rb") as fh:
        magic = fh.read(2)
    is_gz = (magic == b"\x1f\x8b")

    compression = "gzip" if is_gz else None
    df = pd.read_csv(filepath, sep="\t", index_col=0, compression=compression)
    df = df.T
    df.index.name = "sample_id"
    print(f"    Shape (samples x features): {df.shape}")
    return df


def load_survival(filepath: Path) -> pd.DataFrame:
    with open(filepath, "rb") as fh:
        magic = fh.read(2)
    compression = "gzip" if magic == b"\x1f\x8b" else None
    df = pd.read_csv(filepath, sep="\t", compression=compression)
    return df.set_index("sample")


rna_raw     = load_xena_tsv(DATA_DIR / XENA_FILES["rna"]["filename"])
cnv_raw     = load_xena_tsv(DATA_DIR / XENA_FILES["cnv"]["filename"])
meth_raw    = load_xena_tsv(DATA_DIR / XENA_FILES["meth"]["filename"])
survival_df = load_survival(DATA_DIR / XENA_FILES["survival"]["filename"])

print("\nSurvival table shape:", survival_df.shape)
survival_df.head(3)

## 4. Sample Alignment

Keep only **primary tumor samples** (TCGA barcode field 3 starts with `01`) and intersect across all three modalities.

In [ ]:
def is_primary_tumor(barcode: str) -> bool:
    try:
        return barcode.split("-")[3][:2] == "01"
    except IndexError:
        return False


def align_samples(*dataframes) -> list:
    filtered = []
    for df in dataframes:
        primary = [s for s in df.index if is_primary_tumor(s)]
        filtered.append(df.loc[primary])

    common = filtered[0].index
    for df in filtered[1:]:
        common = common.intersection(df.index)

    print(f"Shared primary tumor samples across all modalities: {len(common)}")
    return [df.loc[common] for df in filtered]


rna_aligned, cnv_aligned, meth_aligned = align_samples(rna_raw, cnv_raw, meth_raw)

print(f"  RNA  : {rna_aligned.shape}")
print(f"  CNV  : {cnv_aligned.shape}")
print(f"  Meth : {meth_aligned.shape}")

## 5. Modality-Specific Preprocessing

- **RNA-seq**: drop low-variance / high-missing genes → impute with gene median → z-score per gene  
- **CNV**: drop high-missing genes → impute with 0 (diploid) → drop zero-variance genes  
- **Methylation**: drop high-missing CpGs → impute with site median → M-value transform → clip ±10 → z-score

In [ ]:
def select_top_variable(df: pd.DataFrame, n: int, label: str) -> pd.DataFrame:
    var = df.var(axis=0)
    top = var.nlargest(min(n, df.shape[1])).index
    print(f"  [{label}] Kept {len(top):,} / {df.shape[1]:,} features (top variance)")
    return df[top]


def to_numeric_fast(df: pd.DataFrame) -> pd.DataFrame:
    """Vectorized numeric conversion — avoids slow column-by-column apply."""
    arr = pd.to_numeric(df.values.ravel(), errors="coerce").reshape(df.shape)
    return pd.DataFrame(arr.astype(np.float32), index=df.index, columns=df.columns)


def preprocess_rna(df: pd.DataFrame, n_top: int = 2000) -> pd.DataFrame:
    print("\n[RNA] Preprocessing ...")
    df = df.loc[:, df.isnull().mean(axis=0) < 0.20]
    df = df.fillna(df.median(axis=0))
    df = select_top_variable(df, n_top, "RNA")
    scaler = StandardScaler()
    df = pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)
    print(f"  Final shape: {df.shape}")
    return df.astype(np.float32)


def preprocess_cnv(df: pd.DataFrame, n_top: int = 1000) -> pd.DataFrame:
    """Continuous GISTIC2 log-ratio values — real-valued, Gaussian-compatible for AMP."""
    print("\n[CNV] Preprocessing ...")
    df = to_numeric_fast(df)
    # Impute with 0 (diploid = no copy number change)
    df = df.loc[:, df.isnull().mean(axis=0) < 0.20]
    df = df.fillna(0.0)
    # Drop zero-variance genes (invariant across cohort)
    df = df.loc[:, df.var(axis=0) > 0]
    # Select top highly variable genes
    df = select_top_variable(df, n_top, "CNV")
    # Z-score: continuous log-ratios are approximately Gaussian after centering
    scaler = StandardScaler()
    df = pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)
    print(f"  Final shape: {df.shape}")
    return df.astype(np.float32)


def preprocess_methylation(df: pd.DataFrame, n_top: int = 5000,
                            missing_thresh: float = 0.20) -> pd.DataFrame:
    print("\n[Methylation] Preprocessing ...")

    # 1. Vectorized numeric conversion (avoids slow column-by-column apply)
    print("  Converting to numeric ...")
    df = to_numeric_fast(df)

    # 2. Drop high-missing CpG sites
    df = df.loc[:, df.isnull().mean(axis=0) < missing_thresh]
    print(f"  After missing filter: {df.shape[1]:,} sites")

    # 3. Impute with column median
    #    Use numpy for speed: fill each column median directly on the array
    print("  Imputing missing values ...")
    arr = df.values                                    # (n_samples x n_sites)
    col_medians = np.nanmedian(arr, axis=0)            # one pass, vectorized
    nan_mask = np.isnan(arr)
    arr[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])
    df = pd.DataFrame(arr, index=df.index, columns=df.columns)

    # 4. Select top variable CpG sites on BETA values (before M-transform)
    #    This is the critical step — reduces 485k -> 5k before any heavy math
    df = select_top_variable(df, n_top, "Methylation")

    # 5. M-value transform on the small subset only
    epsilon = 1e-6
    df = df.clip(epsilon, 1 - epsilon)
    df = np.log2(df / (1 - df))
    df = df.clip(-10, 10)

    # 6. Z-score
    scaler = StandardScaler()
    df = pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)
    print(f"  Final shape: {df.shape}")
    return df.astype(np.float32)


X1 = preprocess_rna(rna_aligned,  n_top=TOP_FEATURES["rna"])
X2 = preprocess_cnv(cnv_aligned,  n_top=TOP_FEATURES["cnv"])
X3 = preprocess_methylation(meth_aligned, n_top=TOP_FEATURES["meth"])

assert list(X1.index) == list(X2.index) == list(X3.index), "Sample indices are not aligned!"
sample_ids = X1.index
print(f"\nAll modalities aligned: {len(sample_ids)} samples")
print(f"  X1 RNA         : {X1.shape}")
print(f"  X2 CNV         : {X2.shape}")
print(f"  X3 Methylation : {X3.shape}")

## 6. Low-Rank Decomposition

For each modality, we select the smallest rank explaining 90% of variance, then compute:

$$X_i \approx U_i D_i V_i^\top$$

and record the residual $Z_i = X_i - U_i D_i V_i^\top$.

In [ ]:
def estimate_rank(X: np.ndarray, variance_threshold: float = 0.90) -> int:
    max_components = min(X.shape) - 1
    svd = TruncatedSVD(n_components=max_components, random_state=42)
    svd.fit(X)
    cumvar = np.cumsum(svd.explained_variance_ratio_)
    return int(np.searchsorted(cumvar, variance_threshold)) + 1


def low_rank_decompose(df: pd.DataFrame,
                        rank: int = None,
                        variance_threshold: float = 0.90,
                        name: str = "X") -> dict:
    X = df.values
    n, p = X.shape

    if rank is None:
        rank = estimate_rank(X, variance_threshold)
        print(f"[{name}] Estimated rank = {rank}  ({variance_threshold*100:.0f}% variance threshold)")

    svd = TruncatedSVD(n_components=rank, random_state=42)
    svd.fit(X)

    V = svd.components_.T                  # (p × rank)
    D = np.diag(svd.singular_values_)      # (rank × rank)
    U = X @ V @ np.linalg.inv(D)          # (n × rank)

    X_approx = U @ D @ V.T
    Z = X - X_approx
    sigma_Z = Z.std()
    expl_var = svd.explained_variance_ratio_.sum()

    print(f"[{name}] Explained variance: {expl_var*100:.2f}%   "
          f"sigma_Z: {sigma_Z:.4f}   "
          f"U: {U.shape}  D: {D.shape}  V^T: {V.T.shape}")

    return {
        "name": name,
        "X_full":   pd.DataFrame(X,        index=df.index, columns=df.columns),
        "X_approx": pd.DataFrame(X_approx, index=df.index, columns=df.columns),
        "U":        pd.DataFrame(U,        index=df.index),
        "D":        D,
        "V":        pd.DataFrame(V,        index=df.columns, columns=range(rank)),
        "Z":        pd.DataFrame(Z,        index=df.index, columns=df.columns),
        "sigma_Z":  sigma_Z,
        "rank":     rank,
        "explained_variance": expl_var,
    }


VARIANCE_THRESHOLD = 0.90

res1 = low_rank_decompose(X1, variance_threshold=VARIANCE_THRESHOLD, name="RNA")
res2 = low_rank_decompose(X2, variance_threshold=VARIANCE_THRESHOLD, name="CNV")
res3 = low_rank_decompose(X3, variance_threshold=VARIANCE_THRESHOLD, name="Methylation")

results = [res1, res2, res3]

## 7. Summary

In [ ]:
print(f"{'='*60}")
print(f"FINAL MATRIX SUMMARY   (n = {len(sample_ids)} samples)")
print(f"{'='*60}")
for res in results:
    print(f"\n  {res['name']}")
    print(f"    Full matrix X  : {res['X_full'].shape}")
    print(f"    Rank           : {res['rank']}")
    print(f"    Explained var  : {res['explained_variance']*100:.1f}%")
    print(f"    sigma_Z (noise): {res['sigma_Z']:.4f}")
    print(f"    U shape        : {res['U'].shape}")
    print(f"    V shape        : {res['V'].shape}")

## 8. Diagnostics

In [ ]:
# ---- Singular value scree plots ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, res in zip(axes, results):
    singular_vals = np.diag(res["D"])
    ax.plot(singular_vals, "o-", color="steelblue", markersize=4)
    ax.axvline(res["rank"] - 1, color="red", linestyle="--", label=f"rank={res['rank']}")
    ax.set_title(f"{res['name']} — Singular Values")
    ax.set_xlabel("Component")
    ax.set_ylabel("Singular Value")
    ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "singular_values.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---- Residual distributions ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, res in zip(axes, results):
    z_flat   = res["Z"].values.flatten()
    z_sample = np.random.default_rng(42).choice(z_flat, size=min(50000, len(z_flat)), replace=False)
    ax.hist(z_sample, bins=80, color="salmon", edgecolor="none", density=True)
    xs = np.linspace(z_sample.min(), z_sample.max(), 300)
    ax.plot(xs, stats.norm.pdf(xs, 0, res["sigma_Z"]),
            "k-", linewidth=2, label=f"N(0, {res['sigma_Z']:.3f}²)")
    ax.set_title(f"{res['name']} — Residuals")
    ax.set_xlabel("Residual value")
    ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / "residuals.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Save Matrices

In [ ]:
for res in results:
    name = res["name"].replace(" ", "_")
    res["X_full"].to_csv(OUT_DIR   / f"{name}_X_full.csv")
    res["X_approx"].to_csv(OUT_DIR / f"{name}_X_approx.csv")
    res["U"].to_csv(OUT_DIR        / f"{name}_U.csv")
    np.savetxt(OUT_DIR             / f"{name}_D.csv", res["D"], delimiter=",")
    res["V"].to_csv(OUT_DIR        / f"{name}_V.csv")
    res["Z"].to_csv(OUT_DIR        / f"{name}_Z_residual.csv")
    print(f"[Saved] {name}  →  {OUT_DIR}")

print("\nDone.")
print(f"  X1 RNA        : {X1.shape}")
print(f"  X2 CNV        : {X2.shape}")
print(f"  X3 Methylation: {X3.shape}")